# Dense Reference Vs Operator Solver

This notebook compares the tiny dense Hamiltonian reference against direct Kohn-Sham operator application. The dense path is useful for validation, but it scales badly because it builds an explicit matrix with one row and column per grid point.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mlx_atomistic.benchmarks.dft_operator import build_payload
from mlx_atomistic.dft import (
    DFTSystem,
    DenseHamiltonianReference,
    KohnShamOperator,
    SCFConfig,
    SubspaceDiagonalizer,
    run_scf,
)

On a `2³` grid we can afford to build the whole Hamiltonian. The key correctness check is that a dense matrix-vector product matches `operator.apply_hamiltonian(ψ)`.

In [ ]:
system = DFTSystem.one_center(cell=(4.0, 4.0, 4.0), grid_shape=(2, 2, 2), center=(2.0, 2.0, 2.0))
result = run_scf(system, config=SCFConfig(max_iterations=1, solver="dense", seed=5))
operator = KohnShamOperator.from_density(
    system.grid,
    system.pseudopotential.field(system.grid),
    result.density,
)
reference = DenseHamiltonianReference(operator)
subspace = SubspaceDiagonalizer(tolerance=1e-5).solve(operator, n_orbitals=1)
trial = result.orbitals[0]
dense = np.array(reference.matvec(trial))
applied = np.array(operator.apply_hamiltonian(trial))
{
    "dense_vs_operator_max_error": float(np.max(np.abs(dense - applied))),
    "dense_eigenvalue": float(np.array(reference.diagonalize(1).eigenvalues)[0]),
    "subspace_eigenvalue": float(np.array(subspace.eigenvalues)[0]),
    "subspace_residual": float(np.max(np.array(subspace.residuals))),
}

The benchmark rows show why the dense reference must stay a validation path. Matrix construction and diagonalization grow quickly with grid size, while direct operator application remains the path we eventually optimize.

In [ ]:
payload = build_payload(grid_shape=(2, 2, 2), sizes="2,4", iterations=1)
rows = payload["cases"]
[{key: row[key] for key in ["grid_shape", "operator_apply_ms", "dense_build_ms", "dense_diagonalize_ms"]} for row in rows]

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4), constrained_layout=True)
x = [str(tuple(row["grid_shape"])) for row in rows]
ax.plot(x, [row["operator_apply_ms"] for row in rows], marker="o", label="operator apply")
ax.plot(x, [row["dense_build_ms"] for row in rows], marker="o", label="dense build")
ax.plot(x, [row["dense_diagonalize_ms"] for row in rows], marker="o", label="dense diagonalize")
ax.set_ylabel("ms")
ax.set_title("tiny-grid validation cost")
ax.legend();